# Silver Layer — Clean Data & Feature Extraction

Welcome to the **Silver Layer**. This is where raw pixels become structured game state.

## Where Are We in the Pipeline?

```
Raw Screenshot  ──►  [BRONZE]  ──►  [SILVER]  ──►  [GOLD]  ──►  Prediction
                      (denoise,      (you are here)  (train model)
                       sharpen)
```

The Bronze layer gave us a clean image. Now we need to **extract structured,
game-relevant features** from that image — things like:

- **Player health** (how hurt is the player?)
- **Enemy positions and health** (where are enemies, how hurt are they?)
- **Attack state** (is the player attacking?)
- **Defense state** (is the player blocking/dodging?)
- **Damage indicators** (is the screen showing damage feedback?)

These features are fed to the **Gold layer** (a classifier) which produces the
final `winning`/`losing`/`stalemate` prediction.

---

## Why a CNN Instead of Hardcoded Positions?

A naive approach would be to hardcode pixel coordinates:
```python
player_health_bar = image[20:50, 2000:2500]  # brittle!
```

This breaks if:
- The game UI gets updated
- You switch to a different game
- The resolution changes
- Enemies move to unexpected positions

**Instead, we use a CNN that learns the relationships end-to-end:**
- The CNN sees the whole image
- It learns what a health bar *looks like*, not where it *is*
- It learns that enemy health bars are *above enemy heads*
- It learns that a red flash means *player damage*

This makes the system **game-adaptive** — retrain on new labeled data and it
works with any game.

---

## Setup: Imports & Paths

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch

from src.config import BRONZE_DIR, SILVER_DIR
from src.pipeline.bronze import load_image, preprocess, process_image as bronze_process
from src.pipeline.silver import (
    H, W,
    SilverFeatures,
    process_image as silver_process,
    render_synthetic_frame,
    generate_synthetic_annotation,
)
from src.models.silver_cnn import SilverCNN, SilverOutput, SilverDataset

sns.set_theme(style="darkgrid")
%matplotlib inline

---

# Part 1: Understanding the Multi-Head CNN

## Model Architecture

The `SilverCNN` is a **multi-head CNN** — one shared backbone with several
specialized output heads:

```
                  ┌─── player_health ──────── (scalar 0..1)
                  ├─── player_position ────── (cx, cy normalized)
     ┌────────┐   ├─── enemy_heatmap ──────── (3 × 16 × 28 heatmaps)
     │Backbone│───├─── enemy_health ───────── (3 scalars 0..1)
     └────────┘   ├─── attacking ──────────── (binary 0..1)
                  ├─── defending ──────────── (binary 0..1)
                  └─── damage_indicator ───── (binary 0..1)
```

### Backbone (shared feature extractor)
6 convolutional blocks that progressively downsample the 1440×2560 input
to a 23×40 feature map (stride 64). Every block is Conv → BatchNorm → ReLU.

### Why multiple heads?
Each head learns different features from the same backbone:
- **Player health** needs to find the health bar color/length
- **Enemy heatmap** needs to detect character-shaped blobs
- **Attacking/defending** needs to recognize motion cues and pose

Training all heads jointly allows them to share low-level features
(edges, colors, textures) while specializing at the output.

### Heatmap approach for enemy detection
Instead of bounding box regression, the model outputs **heatmaps** —
2D probability maps at 16×28 resolution. Each of the 3 channels (for up to
3 enemies) peaks at the enemy's location. This is:
- **Faster** than region proposal networks
- **Simple** to train (just MSE on the heatmap)
- **Flexible** — heatmaps can represent fuzzy spatial concepts

In [ ]:
model = SilverCNN()
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

## Test a Forward Pass

Feed a random image through the model to verify the output shapes and ranges.

In [ ]:
x = torch.randn(1, 3, H, W)  # batch of 1, RGB, full resolution
with torch.no_grad():
    out = model(x)

print("Output shapes:")
print(f"  player_health:    {out.player_health.shape}     (scalar 0..1)")
print(f"  player_position:  {out.player_position.shape}    (cx, cy normalized)")
print(f"  enemy_heatmap:    {out.enemy_heatmap.shape}  (3 enemy slots × 16H × 28W)")
print(f"  enemy_health:     {out.enemy_health.shape}      (3 scalars)")
print(f"  attacking:        {out.attacking.shape}         (binary)")
print(f"  defending:        {out.defending.shape}         (binary)")
print(f"  damage_indicator: {out.damage_indicator.shape}  (binary)")

print("\nOutput ranges (should all be 0..1):")
print(f"  player_health:    {out.player_health.item():.4f}")
print(f"  player_position:  ({out.player_position[0,0]:.4f}, {out.player_position[0,1]:.4f})")
print(f"  enemy_health:     {out.enemy_health[0].tolist()}")
print(f"  attacking:        {out.attacking.item():.4f}")
print(f"  defending:        {out.defending.item():.4f}")
print(f"  damage_indicator: {out.damage_indicator.item():.4f}")

# Visualize one of the heatmaps
heatmap_np = out.enemy_heatmap[0, 0].cpu().numpy()
plt.figure(figsize=(6, 4))
plt.imshow(heatmap_np, cmap="hot", aspect="auto")
plt.colorbar(label="Enemy presence probability")
plt.title("Sample Enemy Heatmap (Channel 0)")
plt.xlabel("Width (28 cells)")
plt.ylabel("Height (16 cells)")
plt.tight_layout()
plt.show()

---

# Part 2: Synthetic Data Generation

To train the SilverCNN, we need labeled data: images paired with ground-truth
game state annotations. The `render_synthetic_frame()` and
`generate_synthetic_annotation()` functions create this data programmatically.

Each synthetic frame contains:
- A **player** circle at center screen
- **1-3 enemy** circles at random positions
- **Health bars** above each enemy's head and in the top-right for the player
- Ground-truth labels for health values, positions, attack/defense state

This lets us **test the training pipeline** before collecting real labeled
screenshots. Eventually, real screenshots with human annotations replace
the synthetic data.

In [ ]:
synthetic_img = render_synthetic_frame(num_enemies=2, seed=42)
synthetic_ann = generate_synthetic_annotation(num_enemies=2, seed=42)

print("Synthetic Annotation:")
print(json.dumps(synthetic_ann, indent=2))

fig, ax = plt.subplots(1, 1, figsize=(16, 9))
ax.imshow(synthetic_img)
ax.set_title("Synthetic Training Frame\n(Green = player, Red = enemies)", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

## Generating a Dataset

The `SilverDataset` class pairs images and annotations for PyTorch training.
Here we generate a small synthetic dataset to demonstrate.

In [ ]:
import tempfile
import os

# Create a temporary dataset directory
tmp_dir = Path(tempfile.mkdtemp())
img_dir = tmp_dir / "images"
ann_dir = tmp_dir / "annotations"
img_dir.mkdir(exist_ok=True)
ann_dir.mkdir(exist_ok=True)

# Generate 5 synthetic samples
for i in range(5):
    img = render_synthetic_frame(seed=i)
    ann = generate_synthetic_annotation(seed=i)
    cv2.imwrite(str(img_dir / f"frame_{i:03d}.png"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    with open(ann_dir / f"frame_{i:03d}.json", "w") as f:
        json.dump(ann, f)

ds = SilverDataset(str(tmp_dir))
print(f"Dataset length: {len(ds)}")

x_sample, y_sample = ds[0]
print(f"Image tensor shape: {x_sample.shape}")
print(f"Player health label: {y_sample.player_health.item():.4f}")
print(f"Attacking label: {y_sample.attacking.item()}")
print(f"Defending label: {y_sample.defending.item()}")

# Clean up
import shutil
shutil.rmtree(tmp_dir)

---

# Part 3: Running the Silver Pipeline on Real Screenshots

Now let's process a real screenshot from `data/bronze/` through the Silver layer.
Since we don't have a trained model yet, the pipeline will return **default
features** (all zeros/false). When a trained model is available, pass it with
the `model=` argument.

In [ ]:
# Discover available screenshots
pngs = sorted(BRONZE_DIR.glob("*.png"))
if not pngs:
    raise FileNotFoundError(f"No PNGs found in {BRONZE_DIR}")

real_image_path = str(pngs[0])
print(f"Processing: {pngs[0].name}")

# First run Bronze to get the preprocessed image
bronze_result = bronze_process(real_image_path, str(SILVER_DIR))
print(f"Bronze output: {bronze_result['preprocessed']}")

# Then run Silver (no model = defaults)
silver_result = silver_process(real_image_path, str(SILVER_DIR))
print(f"\nSilver features saved to: {silver_result['features_json']}")

In [ ]:
# Display the features nicely
features = silver_result["silver_features"]

print("=" * 50)
print("SILVER FEATURES")
print("=" * 50)
print(f"Image path         : {features['image_path']}")
print(f"Player health      : {features['player_health']:.3f}")
print(f"Player position    : ({features['player_position'][0]:.3f}, {features['player_position'][1]:.3f})")
print(f"Number of enemies  : {features['num_enemies']}")
print(f"Attacking          : {features['attacking']}")
print(f"Defending          : {features['defending']}")
print(f"Damage indicator   : {features['damage_indicator']}")

if features['enemies']:
    print("\nEnemy Details:")
    for i, enemy in enumerate(features['enemies']):
        print(f"  Enemy {i+1}:")
        print(f"    BBox       : {enemy['bbox']}")
        print(f"    Health     : {enemy['health']:.3f}")
        print(f"    Health bar : {enemy['health_bar_bbox']}")
        print(f"    Confidence : {enemy['confidence']:.3f}")
else:
    print("\n(No enemies detected — model not trained yet)")

## Visualize the Silver Features on the Image

Let's overlay the extracted features back onto the screenshot so you can see
what the Silver layer "sees".

In [ ]:
def draw_silver_features(image, features):
    """Draw SilverFeatures as colored overlays on the image."""
    vis = image.copy()
    
    # Player position (center)
    cx = int(features['player_position'][0] * W)
    cy = int(features['player_position'][1] * H)
    cv2.circle(vis, (cx, cy), 50, (0, 255, 0), 3)
    cv2.putText(vis, "PLAYER", (cx - 40, cy - 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    
    # Player health text
    health_pct = features['player_health'] * 100
    cv2.putText(vis, f"Health: {health_pct:.0f}%", (W - 550, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    
    # Enemies
    for i, enemy in enumerate(features['enemies']):
        bx, by, bw, bh = enemy['bbox']
        cv2.rectangle(vis, (bx, by), (bx + bw, by + bh), (255, 0, 0), 3)
        cv2.putText(vis, f"ENEMY {i+1}", (bx, by - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
        
        # Health bar
        hx, hy, hw, hh = enemy['health_bar_bbox']
        fill = int(hw * enemy['health'])
        cv2.rectangle(vis, (hx, hy), (hx + hw, hy + hh), (100, 100, 100), -1)
        color = (0, 255, 0) if enemy['health'] > 0.5 else (0, 0, 255)
        cv2.rectangle(vis, (hx, hy), (hx + fill, hy + hh), color, -1)
    
    # State indicators
    y_offset = 120
    if features['attacking']:
        cv2.putText(vis, "STATE: ATTACKING", (W - 550, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 165, 255), 2)
        y_offset += 40
    if features['defending']:
        cv2.putText(vis, "STATE: DEFENDING", (W - 550, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)
        y_offset += 40
    if features['damage_indicator']:
        cv2.putText(vis, "DAMAGE TAKEN", (W - 550, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
    
    return vis


# Load the preprocessed Bronze image
bronze_img = cv2.imread(bronze_result['preprocessed'])
bronze_img = cv2.cvtColor(bronze_img, cv2.COLOR_BGR2RGB)

# Overlay features
vis = draw_silver_features(bronze_img, features)

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
axes[0].imshow(bronze_img)
axes[0].set_title("Bronze (Preprocessed)", fontsize=14, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(vis)
axes[1].set_title("Silver — Extracted Features Overlay", fontsize=14, fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

---

# Part 4: Training the SilverCNN (How-To)

Once you have labeled data (either synthetic or real), training the SilverCNN
follows this workflow:

```python
from src.models.silver_cnn import SilverCNN, SilverDataset, compute_loss
from torch.utils.data import DataLoader

# 1. Load dataset
ds = SilverDataset("path/to/labeled_data/")
loader = DataLoader(ds, batch_size=2, shuffle=True)

# 2. Create model and optimizer
model = SilverCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 3. Train
for images, targets in loader:
    outputs = model(images)
    loss = compute_loss(outputs, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

The multi-task loss combines:
- **MSE** for regression heads (health, position, heatmap)
- **Binary cross-entropy** for classification heads (attacking, defending, damage)

All heads are trained simultaneously so they share backbone features.

In [ ]:
# Quick sanity check: compute loss on random data to ensure it backpropagates
model = SilverCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

x = torch.randn(2, 3, H, W)
y = SilverOutput(
    player_health=torch.rand(2),
    player_position=torch.rand(2, 2),
    enemy_heatmap=torch.rand(2, 3, 16, 28),
    enemy_health=torch.rand(2, 3),
    attacking=torch.randint(0, 2, (2,)).float(),
    defending=torch.randint(0, 2, (2,)).float(),
    damage_indicator=torch.randint(0, 2, (2,)).float(),
)

out = model(x)
loss = (
    torch.nn.functional.mse_loss(out.player_health, y.player_health)
    + torch.nn.functional.mse_loss(out.player_position, y.player_position)
    + torch.nn.functional.mse_loss(out.enemy_heatmap, y.enemy_heatmap)
    + torch.nn.functional.mse_loss(out.enemy_health, y.enemy_health)
    + torch.nn.functional.binary_cross_entropy(out.attacking, y.attacking)
    + torch.nn.functional.binary_cross_entropy(out.defending, y.defending)
    + torch.nn.functional.binary_cross_entropy(out.damage_indicator, y.damage_indicator)
)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print(f"✓ Training step completed. Loss: {loss.item():.4f}")

# Verify gradients flowed
grad_norm = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
print(f"✓ Gradient norm: {grad_norm:.4f} (should be > 0)")

---

# Part 5: Running with a Trained Model

When you have a trained checkpoint, load it and pass to `process_image()`:

```python
model = SilverCNN()
model.load_state_dict(torch.load("path/to/model.pth"))
model.eval()

result = silver_process("screenshot.png", "data/silver/", model=model)
features = result["silver_features"]
```

The trained model will output meaningful values in all fields instead of defaults.

---

# Part 6: How Silver Feeds Into Gold

The output of the Silver layer is a structured JSON file containing
`SilverFeatures`. This JSON is consumed by the **Gold layer** (`gold.py`)
which trains an ML classifier to predict the final game state:

## Gold Model Inputs (from Silver)

| Feature | Type | Purpose for Classification |
|---------|------|---------------------------|
| `player_health` | float 0-1 | Low health → losing |
| `num_enemies` | int | More enemies → losing |
| `enemies[i].health` | float 0-1 | Low enemy health → winning |
| `attacking` | bool | Aggression → winning/stalemate |
| `defending` | bool | Defense → losing/stalemate |
| `damage_indicator` | bool | Taking damage → losing |

## Gold Output
```
winning | losing | stalemate
```

---

## Summary

The Silver layer converts preprocessed Bronze images into **structured game
features** using a multi-head CNN. Key takeaways:

- **CNN instead of hardcoded rules** — adapts to UI changes and different games
- **Multi-head architecture** — one backbone, seven output heads for different tasks
- **Heatmap-based enemy detection** — simple, flexible, differentiable
- **Synthetic data** enables pipeline testing before real labels exist
- **Output JSON** feeds directly into the Gold classifier

Next step: open **`03_gold_modeling.ipynb`** to train the final classifier.